In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

def remap_mask(mask):
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

# Dataset Class
class SUIMDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_size = image_size

        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image
        img_path = os.path.join(self.image_dir, self.images[idx])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.image_size, self.image_size))
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        # Load mask
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.image_size, self.image_size),
                          interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy(mask)

        # Remap mask labels
        mask = remap_mask(mask)

        return image, mask

# Dataset Paths
BASE_PATH = "/kaggle/input/q3-stage3-2026/dataset"

image_dir = os.path.join(BASE_PATH, "images")
mask_dir  = os.path.join(BASE_PATH, "masks")

print("Images exist:", os.path.exists(image_dir))
print("Masks exist :", os.path.exists(mask_dir))

# Dataset & DataLoader
train_dataset = SUIMDataset(image_dir, mask_dir)
train_loader  = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Display Images & Masks
def show_samples(dataset, num_samples=3):
    plt.figure(figsize=(12, 4 * num_samples))

    for i in range(num_samples):
        image, mask = dataset[i]

        plt.subplot(num_samples, 2, i * 2 + 1)
        plt.title("Image")
        plt.imshow(image.permute(1, 2, 0))
        plt.axis("off")

        plt.subplot(num_samples, 2, i * 2 + 2)
        plt.title("Mask (Remapped Classes)")
        plt.imshow(mask, cmap="tab10")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Show 3 samples
show_samples(train_dataset, num_samples=3)

In [ ]:
import torch.nn as nn
import segmentation_models_pytorch as smp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 8

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
)

model = model.to(DEVICE)
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

images, masks = next(iter(train_loader))
images = images.to(DEVICE)

outputs = model(images)


In [ ]:
# Training function
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

# Validation function
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item()

    return running_loss / len(loader)

# Training Configuration
EPOCHS = 10
train_loader = train_loader
val_loader   = train_loader

# Training Loop
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss = validate_one_epoch(model, val_loader, criterion, DEVICE)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
# Loss Function + Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training Configuration
EPOCHS = 10

train_losses = []
val_losses = []

# Training Loop
for epoch in range(EPOCHS):
    # Training
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    # Validation
    val_loss = validate_one_epoch(model, val_loader, criterion, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# Plot Loss Curve
plt.figure(figsize=(8,6))
plt.plot(range(1, EPOCHS+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, EPOCHS+1), val_losses, label="Validation Loss", marker='x')
plt.title("Training and Validation Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
def visualize_predictions(model, dataset, device, num_samples=5):
    model.eval()

    plt.figure(figsize=(12, num_samples * 4))

    for i in range(num_samples):
        image, mask = dataset[i]
        image_input = image.unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(image_input)
            pred_mask = torch.argmax(pred, dim=1).squeeze().cpu()

        # Plot Image
        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(image.permute(1, 2, 0))
        plt.title("Image")
        plt.axis("off")

        # Plot Ground Truth Mask
        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(mask, cmap="tab10")
        plt.title("Ground Truth")
        plt.axis("off")

        # Plot Predicted Mask
        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(pred_mask, cmap="tab10")
        plt.title("Predicted Mask")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(model, train_dataset, DEVICE, num_samples=5)
